# Category 1: Single Node Neural Network
In this section, we create a basic building block of a neural network: a single neuron.
A neuron takes inputs, multiplies them by weights, adds a bias, and passes the result through an activation function (like the sigmoid function) to produce an output.


In [ ]:
import numpy as np

def sigmoid(x):
    """
    Sigmoid activation function: f(x) = 1 / (1 + e^(-x))
    Maps any input to a value between 0 and 1.
    """
    return 1 / (1 + np.exp(-x))

def deriv_sigmoid(x):
    """
    Derivative of the sigmoid function: f'(x) = f(x) * (1 - f(x))
    Used later for backpropagation during training.
    """
    fx = sigmoid(x)
    return fx * (1 - fx)


In [39]:
class Neuron: 
    """
    A single neuron with a specific set of weights and a bias.
    """
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

    def feedforward(self, inputs):
        """
        Calculates the output of the neuron.
        Step 1: Dot product of weights and inputs, plus bias.
        Step 2: Pass the result through the sigmoid activation function.
        """
        total = np.dot(self.weights, inputs) + self.bias
        return sigmoid(total)


In [40]:
# --- Testing the Single Neuron ---

# Set arbitrary weights and bias
weights = np.array([0, 1])
bias = 4

# Initialize the neuron
n = Neuron(weights, bias)

# Provide an input and calculate the feedforward output
x = np.array([2, 3])
output = n.feedforward(x)

print(f"Output of the single neuron: {output:.4f}")


Output of the single neuron: 0.9991


# Category 2: Multi-Layer Neural Network & Training
Here, we connect multiple neurons to form a network. This network consists of:
- An input layer (2 features: Weight and Height)
- A hidden layer (2 neurons: h1, h2)
- An output layer (1 neuron: o1, predicting gender)

We also introduce a small dataset and train the network using Mean Squared Error (MSE) loss and Backpropagation.


In [41]:
import pandas as pd

# --- 1. Create a Small Dataset ---
data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'Weight(lb)': [133, 160, 152, 120],
    'Height(in)': [65, 72, 70, 60],
    'Gender' : ['F', 'M', 'M', 'F']
}
df = pd.DataFrame(data)
print("--- Original Data ---")
print(df)

# --- 2. Preprocess Data ---
# Shift data by subtracting the mean to center it around 0.
mean_weight = df['Weight(lb)'].mean()
mean_height = df['Height(in)'].mean()

df['Weight(lb)'] -= mean_weight
df['Height(in)'] -= mean_height

# Map 'F' to 1 and 'M' to 0 for binary classification
df['Gender'] = df['Gender'].map({'F': 1, 'M': 0})

print("\n--- Preprocessed Data ---")
print(df)

# Extract numpy arrays for training
training_data = df[['Weight(lb)', 'Height(in)']].values
training_labels = df['Gender'].values


--- Original Data ---
      Name  Weight(lb)  Height(in) Gender
0    Alice         133          65      F
1      Bob         160          72      M
2  Charlie         152          70      M
3    Diana         120          60      F

--- Preprocessed Data ---
      Name  Weight(lb)  Height(in)  Gender
0    Alice       -8.25       -1.75       1
1      Bob       18.75        5.25       0
2  Charlie       10.75        3.25       0
3    Diana      -21.25       -6.75       1


In [42]:
def mse_loss(y_true, y_pred):
    """
    Calculates the Mean Squared Error (MSE) loss.
    Measures the average squared difference between true labels and predictions.
    """
    return ((y_true - y_pred) ** 2).mean()


In [43]:
class OurNeuralNetwork:
    """
    A neural network with:
    - 2 inputs
    - a hidden layer with 2 neurons (h1, h2)
    - an output layer with 1 neuron (o1)
    """
    def __init__(self):
        # Initialize Weights randomly
        self.w1 = np.random.normal()
        self.w2 = np.random.normal()
        self.w3 = np.random.normal()
        self.w4 = np.random.normal()
        self.w5 = np.random.normal()
        self.w6 = np.random.normal()

        # Initialize Biases randomly
        self.b1 = np.random.normal()
        self.b2 = np.random.normal()
        self.b3 = np.random.normal()

    def feedforward(self, x):
        """
        Passes inputs through the network to get a prediction.
        """
        h1 = sigmoid(self.w1 * x[0] + self.w2 * x[1] + self.b1)
        h2 = sigmoid(self.w3 * x[0] + self.w4 * x[1] + self.b2)
        o1 = sigmoid(self.w5 * h1 + self.w6 * h2 + self.b3)
        return o1

    def train(self, data, all_y_trues, epochs=1000):
        """
        Trains the network using Gradient Descent.
        """
        learn_rate = 0.01

        for epoch in range(epochs):
            for x, y_true in zip(data, all_y_trues):
                # --- Forward Pass ---
                sum_h1 = self.w1 * x[0] + self.w2 * x[1] + self.b1
                h1 = sigmoid(sum_h1)

                sum_h2 = self.w3 * x[0] + self.w4 * x[1] + self.b2
                h2 = sigmoid(sum_h2)

                sum_o1 = self.w5 * h1 + self.w6 * h2 + self.b3
                o1 = sigmoid(sum_o1)
                y_pred = o1

                # --- Backpropagation (Calculating Gradients) ---
                d_L_d_ypred = -2 * (y_true - y_pred)

                # Output neuron (o1)
                d_ypred_d_w5 = h1 * deriv_sigmoid(sum_o1)
                d_ypred_d_w6 = h2 * deriv_sigmoid(sum_o1)
                d_ypred_d_b3 = deriv_sigmoid(sum_o1)

                d_ypred_d_h1 = self.w5 * deriv_sigmoid(sum_o1)
                d_ypred_d_h2 = self.w6 * deriv_sigmoid(sum_o1)

                # Hidden neuron (h1)
                d_h1_d_w1 = x[0] * deriv_sigmoid(sum_h1)
                d_h1_d_w2 = x[1] * deriv_sigmoid(sum_h1)
                d_h1_d_b1 = deriv_sigmoid(sum_h1)

                # Hidden neuron (h2)
                d_h2_d_w3 = x[0] * deriv_sigmoid(sum_h2)
                d_h2_d_w4 = x[1] * deriv_sigmoid(sum_h2)
                d_h2_d_b2 = deriv_sigmoid(sum_h2)

                # --- Gradient Descent (Updating Weights and Biases) ---
                self.w1 -= learn_rate * d_L_d_ypred * d_ypred_d_h1 * d_h1_d_w1
                self.w2 -= learn_rate * d_L_d_ypred * d_ypred_d_h1 * d_h1_d_w2
                self.b1 -= learn_rate * d_L_d_ypred * d_ypred_d_h1 * d_h1_d_b1

                self.w3 -= learn_rate * d_L_d_ypred * d_ypred_d_h2 * d_h2_d_w3
                self.w4 -= learn_rate * d_L_d_ypred * d_ypred_d_h2 * d_h2_d_w4
                self.b2 -= learn_rate * d_L_d_ypred * d_ypred_d_h2 * d_h2_d_b2

                self.w5 -= learn_rate * d_L_d_ypred * d_ypred_d_w5
                self.w6 -= learn_rate * d_L_d_ypred * d_ypred_d_w6
                self.b3 -= learn_rate * d_L_d_ypred * d_ypred_d_b3

            # --- Evaluate Loss periodically ---
            if epoch % 100 == 0:
                y_preds = np.apply_along_axis(self.feedforward, 1, data)
                loss = mse_loss(all_y_trues, y_preds)
                print(f"Epoch {epoch:4d} loss: {loss:.4f}")


In [44]:
# --- Train the Neural Network ---
print("Training the network on the small dataset...")
network = OurNeuralNetwork()
network.train(training_data, training_labels, epochs=1000)


Training the network on the small dataset...
Epoch    0 loss: 0.3249
Epoch  100 loss: 0.2678
Epoch  200 loss: 0.2101
Epoch  300 loss: 0.1607
Epoch  400 loss: 0.1235
Epoch  500 loss: 0.0969
Epoch  600 loss: 0.0780
Epoch  700 loss: 0.0642
Epoch  800 loss: 0.0540
Epoch  900 loss: 0.0462


In [45]:
# --- Making Predictions ---
# Let's test the network on some new (unseen) data

def predict_gender(network, weight, height, mean_weight, mean_height):
    """ Helper function to preprocess input and predict. """
    # Center the data using the SAME mean from training
    x = np.array([weight - mean_weight, height - mean_height])
    pred = network.feedforward(x)
    predicted_class = "Female" if pred >= 0.5 else "Male"
    print(f"Weight: {weight} lbs, Height: {height} in -> Probability (Female): {pred:.4f} => {predicted_class}")

predict_gender(network, weight=128, height=63, mean_weight=mean_weight, mean_height=mean_height)  # Emily
predict_gender(network, weight=156, height=72, mean_weight=mean_weight, mean_height=mean_height)  # Anshumaan


Weight: 128 lbs, Height: 63 in -> Probability (Female): 0.7629 => Female
Weight: 156 lbs, Height: 72 in -> Probability (Female): 0.1548 => Male


# Category 3: Training on GPU using PyTorch
In this section, we use PyTorch to train our network on the GPU.
This allows us to handle much larger datasets (like our 3000-sample dataset) and perform computations in parallel using CUDA.


In [46]:
# --- 0. Importing Libraries and Device Setup ---

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")


Training on: cuda


In [47]:
# --- 1. Load & Preprocess Data ---

df = pd.read_csv('archive/Training set.csv')

# --- NEW: Data Cleaning (Removing Outliers) ---
# The dataset contains errors (e.g., a person listed as 3050 cm / 100 feet tall).
# These outliers skew the mean and standard deviation, crippling the network.
# We filter the dataset to only include physically realistic human measurements:
# Height between 100cm and 250cm, Weight between 30kg and 150kg.
df = df[(df['Height'] > 100) & (df['Height'] < 250)]
df = df[(df['Weight'] > 30) & (df['Weight'] < 150)]

# Calculate the mean and standard deviation for Z-score normalization
height_mean = df['Height'].mean()
height_std  = df['Height'].std()

weight_mean = df['Weight'].mean()
weight_std  = df['Weight'].std()

# CRITICAL FIX: Z-Score Normalization
# We MUST divide by standard deviation, not just subtract the mean.
# Without dividing by std, the huge inputs (e.g. 160cm) create massive numbers
# that cause the Sigmoid function gradients to vanish immediately,
# resulting in the network getting "stuck" and outputting a constant number (like 0.19).
df['Height'] = (df['Height'] - height_mean) / height_std
df['Weight'] = (df['Weight'] - weight_mean) / weight_std

df['Sex'] = df['Sex'].map({'Female': 1, 'Male': 0})

X = df[['Height', 'Weight']].values.astype(np.float32)
y = df['Sex'].values.astype(np.float32)

X_tensor = torch.tensor(X).to(device)
y_tensor  = torch.tensor(y).unsqueeze(1).to(device)


In [48]:
# --- 2. Network Architecture ---

class GPUNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, x):
        x = torch.sigmoid(self.hidden(x))
        x = torch.sigmoid(self.output(x))
        return x

model = GPUNeuralNetwork().to(device)
print(model)


GPUNeuralNetwork(
  (hidden): Linear(in_features=2, out_features=16, bias=True)
  (output): Linear(in_features=16, out_features=1, bias=True)
)


In [ ]:
# --- 3. DataLoader (Batching) ---

dataset = TensorDataset(X_tensor, y_tensor)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)


In [35]:
# --- 4. Loss & Optimizer ---

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [36]:
# --- 5. Training Loop ---

EPOCHS = 100

for epoch in range(EPOCHS):
    model.train() 
    total_loss = 0
    
    for X_batch, y_batch in loader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if epoch % 10 == 0:
        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch:3d} | Loss: {avg_loss:.4f}")


Epoch   0 | Loss: 0.3015
Epoch  10 | Loss: 0.3008
Epoch  20 | Loss: 0.3014
Epoch  30 | Loss: 0.3009
Epoch  40 | Loss: 0.3006
Epoch  50 | Loss: 0.3009
Epoch  60 | Loss: 0.3003
Epoch  70 | Loss: 0.3010
Epoch  80 | Loss: 0.3004
Epoch  90 | Loss: 0.3009


In [37]:
# --- 6. Inference & Accuracy ---

model.eval() 
with torch.no_grad(): 
    
    preds = model(X_tensor)
    predicted_labels = (preds >= 0.5).float()
    accuracy = (predicted_labels == y_tensor).float().mean()
    print(f"Overall Accuracy: {accuracy.item() * 100:.2f}%\n")

    # CRITICAL FIX: Proper Units and Order
    # The dataset uses Centimeters (cm) and Kilograms (kg).
    # Emily is 63 inches (160 cm) and 128 lbs (58 kg).
    # The network expects [Height, Weight] in that exact order!
    
    emily_height_cm = 160.0
    emily_weight_kg = 58.0
    
    # We MUST apply the exact same Z-score normalization we did to the training data
    emily_h_scaled = (emily_height_cm - height_mean) / height_std
    emily_w_scaled = (emily_weight_kg - weight_mean) / weight_std
    
    emily = torch.tensor([[emily_h_scaled, emily_w_scaled]], dtype=torch.float32).to(device)
    
    prob = model(emily).item()
    print(f"Emily (160cm, 58kg) -> P(Female): {prob:.4f} => {'Female' if prob >= 0.5 else 'Male'}")

    # --- Testing Anshumaan ---
    anshumaan_height_cm = 183.0
    anshumaan_weight_kg = 72.0
    
    # Normalize using the SAME mean and std as the training data
    anshumaan_h_scaled = (anshumaan_height_cm - height_mean) / height_std
    anshumaan_w_scaled = (anshumaan_weight_kg - weight_mean) / weight_std
    
    anshumaan = torch.tensor([[anshumaan_h_scaled, anshumaan_w_scaled]], dtype=torch.float32).to(device)
    
    prob_anshumaan = model(anshumaan).item()
    print(f"Anshumaan (183cm, 72kg) -> P(Female): {prob_anshumaan:.4f} => {'Female' if prob_anshumaan >= 0.5 else 'Male'}")



Overall Accuracy: 86.97%

Emily (160cm, 58kg) -> P(Female): 0.6747 => Female
Anshumaan (183cm, 72kg) -> P(Female): 0.0140 => Male


In [49]:
# --- 7. Professional Evaluation on Unseen Test Data ---

# In a real Machine Learning project, you NEVER evaluate your final model 
# on the data it trained on (because it might have just memorized the answers).
# You evaluate it on a completely unseen "Test Set".

print("\n--- Running Evaluation on Test Set ---")

# 1. Load the unseen Test dataset
test_df = pd.read_csv('archive/Test set.csv')

# 2. CRITICAL ML RULE: Preprocess using TRAINING parameters!
# Do NOT calculate a new mean or standard deviation for the test set.
# You must transform the test data using the exact same scaling factors you used to train the network.
test_df['Height'] = (test_df['Height'] - height_mean) / height_std
test_df['Weight'] = (test_df['Weight'] - weight_mean) / weight_std

# 3. Encode the labels
test_df['Sex'] = test_df['Sex'].map({'Female': 1, 'Male': 0})

# 4. Convert to tensors and move to GPU
X_test = test_df[['Height', 'Weight']].values.astype(np.float32)
y_test = test_df['Sex'].values.astype(np.float32)

X_test_tensor = torch.tensor(X_test).to(device)
y_test_tensor = torch.tensor(y_test).unsqueeze(1).to(device)

# 5. Evaluate the model
model.eval() # Ensure model is in evaluation mode
with torch.no_grad(): # Disable gradient tracking to save memory
    
    # Get predictions for the entire test set
    test_preds = model(X_test_tensor)
    
    # Calculate Test Loss
    test_loss = loss_fn(test_preds, y_test_tensor)
    
    # Calculate Test Accuracy
    test_predicted_labels = (test_preds >= 0.5).float()
    test_accuracy = (test_predicted_labels == y_test_tensor).float().mean()
    
    print(f"Test Loss: {test_loss.item():.4f}")
    print(f"Test Accuracy: {test_accuracy.item() * 100:.2f}%")
    
    # The Test Accuracy is your "True" accuracy. If it is similar to your 
    # Training Accuracy (~87%), it means your model generalizes well and is not overfitting!




--- Running Evaluation on Test Set ---
Test Loss: 0.7150
Test Accuracy: 51.22%
